<a href="https://colab.research.google.com/github/76213869-sketch/Sem7_MLP2/blob/main/Fase_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FASE 5: EVALUACIÓN COMPLETA + IMPORTANCIA DE VARIABLES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score,
    average_precision_score, ConfusionMatrixDisplay
)

from sklearn.inspection import permutation_importance

# ⚠️ REQUIERE QUE YA HAYAS EJECUTADO FASE 4

# =============================
# 1) PREDICCIONES
# =============================
y_pred_proba = mlp.predict_proba(X_test_scaled)[:, 1]

THRESHOLD = 0.40
y_pred = (y_pred_proba >= THRESHOLD).astype(int)

print("=" * 55)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 55)
print(classification_report(y_test, y_pred))

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"AUC-PR : {average_precision_score(y_test, y_pred_proba):.4f}")

# =============================
# 2) GRÁFICOS DE EVALUACIÓN
# =============================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- MATRIZ DE CONFUSIÓN ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Dropout', 'Dropout'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de Confusión")

# --- PRECISION-RECALL ---
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(recall, precision, linewidth=2)
axes[1].set_title("Curva Precision-Recall")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].grid(alpha=0.3)

# --- HISTOGRAMA ---
axes[2].hist(y_pred_proba[y_test == 0], bins=40, alpha=0.6, label='No Dropout')
axes[2].hist(y_pred_proba[y_test == 1], bins=40, alpha=0.6, label='Dropout')
axes[2].axvline(x=THRESHOLD, linestyle='--')
axes[2].legend()
axes[2].set_title("Distribución de Probabilidades")

plt.tight_layout()
plt.savefig('fase5_evaluacion.png', dpi=150)
plt.show()

# ============================================================
# 3) IMPORTANCIA DE VARIABLES (PERMUTATION)
# ============================================================

result = permutation_importance(
    mlp,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='recall'
)

feat_import = pd.DataFrame({
    'Feature': features,
    'Importance': result.importances_mean,
    'Std': result.importances_std
}).sort_values('Importance', ascending=True)

# --- GRÁFICO ---
plt.figure(figsize=(10, 8))

colors = [
    '#e74c3c' if val > feat_import['Importance'].quantile(0.75)
    else '#3498db' if val > 0
    else '#bdc3c7'
    for val in feat_import['Importance']
]

plt.barh(
    feat_import['Feature'],
    feat_import['Importance'],
    color=colors,
    xerr=feat_import['Std'],
    alpha=0.85,
    capsize=3
)

plt.axvline(x=0, linewidth=1)
plt.title("Importancia de Variables (Permutation Importance)")
plt.xlabel("Impacto en Recall")
plt.tight_layout()
plt.savefig('fase5_importancia_vars.png', dpi=150)
plt.show()

# =============================
# 4) TOP VARIABLES
# =============================
print("\nTOP 5 variables más importantes:")
print(
    feat_import
    .sort_values('Importance', ascending=False)
    .head(5)
    .to_string(index=False)
)